# 02 - Methods: Baseline, Shaping, Curriculum, Imitation

Goal: understand what each project agent is testing. This notebook is mostly explanatory, with small optional runs.

In [ ]:
from pathlib import Path
import sys

def _add_project_root_to_path():
    for base in (Path.cwd(), *Path.cwd().parents):
        for candidate in (base, base / "soccer-twos-starter"):
            if (candidate / "soccer_twos_project" / "notebook_tools.py").exists():
                if str(candidate) not in sys.path:
                    sys.path.insert(0, str(candidate))
                return candidate
    raise FileNotFoundError("Could not find the soccer-twos-starter project root.")

_add_project_root_to_path()

import importlib
import soccer_twos_project.notebook_tools as notebook_tools
importlib.reload(notebook_tools)
from soccer_twos_project.notebook_tools import *

ctx = setup_project()
show_hardware()

## Training Stages

`ppo_baseline` is the comparison point. `ppo_shaped` is the reward-modification experiment. `ppo_curriculum` changes start states over time and is the main performance candidate. `ppo_selfplay` is a fallback if curriculum is not strong enough. `dqn_baseline` is optional for algorithm comparison.

In [ ]:
from soccer_twos_project.training import STAGES, build_training_config
from soccer_twos_project.config import select_profile

profile = select_profile("auto", smoke=True)
print_json(STAGES)
print("Smoke profile:", profile)

## Baseline PPO

Baseline PPO trains a single controlled player against a still opponent policy in `team_vs_policy`. It is useful because it gives a clean curve to compare against the shaped and curriculum variants.

In [ ]:
baseline_config = build_training_config("ppo_baseline", profile)
print_json({"model": baseline_config["model"], "env_config": baseline_config["env_config"]})

## Reward-Shaped PPO

The reward-shaped agent adds a small clipped bonus for useful intermediate behavior. This directly satisfies the reward-modification criterion, and the report should compare its learning curve against baseline PPO.

In [ ]:
shaped_config = build_training_config("ppo_shaped", profile)
print_json(shaped_config["env_config"].get("reward_shaping"))

## Curriculum PPO

Curriculum training starts from easier ball/player placements and advances when reward crosses a threshold. This is expected to learn faster than sparse-reward baseline training.

In [ ]:
from soccer_twos_project.training import load_curriculum

for idx, task in enumerate(load_curriculum()):
    print(idx, task["name"], task["config_fn"])

## Optional Small Runs

Uncomment one small run at a time if you want to see each method create logs before launching the full training notebook.

In [ ]:
# run_training(ctx, "ppo_shaped", profile_name="auto", timesteps=25_000, smoke=True, verbose=1)
# run_training(ctx, "ppo_curriculum", profile_name="auto", timesteps=25_000, smoke=True, verbose=1)
# run_training(ctx, "dqn_baseline", profile_name="auto", timesteps=25_000, smoke=True, verbose=1)

## Imitation Learning

Behavior cloning collects `(observation, action)` pairs from an expert agent, then trains a classifier to imitate the expert. Use this after exporting `soccer_ppo_curriculum`, or use `ceia_baseline_agent` if downloaded.

In [ ]:
# from soccer_twos_project.imitation import collect_dataset, train_bc
# DATASET_PATH = ctx.dirs["datasets"] / "bc_expert_dataset.npz"
# collect_dataset(SimpleNamespace(expert_module="soccer_ppo_curriculum", samples=10_000, output=str(DATASET_PATH), base_port=None, artifact_root=str(ctx.artifact_root)))
# train_bc(SimpleNamespace(dataset=str(DATASET_PATH), agent_name="soccer_bc_imitation", author="Your Name", email="your.email@gatech.edu", description="Behavior cloning Soccer-Twos agent.", hidden_layers="256,256", epochs=10, batch_size=256, lr=1e-3, val_fraction=0.1, seed=0, output_dir=None, artifact_root=str(ctx.artifact_root), no_zip=False))